# MNIST neural-network model selection

This notebook explores activation and initialization, optimization method, architecture, and model evaluation for MNIST classification. All training calls use ridge regularization through `method="Ridge"`. The learning rate and ridge penalty of the single-wide-layer network are selected with stratified cross-validation before the final confusion-matrix comparison.

In [6]:
# Imports
import importlib
import time
import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import ConfusionMatrixDisplay, auc, roc_curve
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from tensorflow.keras.datasets import mnist
import neural_network_functions as nnf
importlib.reload(nnf)

<module 'neural_network_functions' from 'c:\\Users\\ktann\\OneDrive - Universitetet i Oslo\\UiO\\Høst 2026\\fys-stk3155\\FYS-STK3155---Projects---UiO\\Code\\neural_network_functions.py'>

## Configuration and data preparation

Set `USE_SMALL_DATASET = True` for a quick run with 1,000 training images and 100 test images. The full setting uses all 60,000 training images and 10,000 test images. Pixel values are scaled to `[0, 1]`; integer labels are retained for scoring and one-hot targets are used for softmax cross-entropy training.

In [7]:
USE_SMALL_DATASET = False
SMALL_TRAIN_SIZE = 1_000
SMALL_TEST_SIZE = 100
EPOCHS = 20
BATCH_SIZE = 128 
LEARNING_RATE = 0.001
RIDGE_LAMBDA = 0.0

(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

if USE_SMALL_DATASET:
    _, X_train_raw, _, y_train_raw = train_test_split(
        X_train_raw, y_train_raw, test_size=SMALL_TRAIN_SIZE,
        random_state=42, stratify=y_train_raw,
    )
    _, X_test_raw, _, y_test_raw = train_test_split(
        X_test_raw, y_test_raw, test_size=SMALL_TEST_SIZE,
        random_state=42, stratify=y_test_raw,
    )

X_train = X_train_raw.reshape(len(X_train_raw), 784).astype(float) / 255.0
X_test = X_test_raw.reshape(len(X_test_raw), 784).astype(float) / 255.0
y_train = np.asarray(y_train_raw).astype(int)
y_test = np.asarray(y_test_raw).astype(int)
Y_train = np.eye(10)[y_train]
Y_test = np.eye(10)[y_test]

print("Train:", X_train.shape, Y_train.shape)
print("Test:", X_test.shape, Y_test.shape)
print("Range:", X_train.min(), X_train.max())

Train: (60000, 784) (60000, 10)
Test: (10000, 784) (10000, 10)
Range: 0.0 1.0


## Experiment 1: Activation function and weight initialization

**Fixed:** architecture `784 → 100 → 100 → 10`, Adam, learning rate, batch size, epochs, softmax output, cross-entropy loss, and ridge training.

**Varied:** hidden-layer activation and initialization. Within each initialization, activation functions begin from identical parameter copies.

In [ ]:
initializations = ["LeCun", "Xavier", "He"]
activations = {
    "Sigmoid": (nnf.sigmoid, nnf.sigmoid_der),
    "ReLU": (nnf.ReLU, nnf.ReLU_der),
    "Leaky ReLU": (nnf.leaky_ReLU, nnf.leaky_ReLU_der),
}
activation_results = []

def output_derivative(z):
    return np.ones_like(z)

for initialization in initializations:
    np.random.seed(42)
    initial_layers = nnf.create_layers(784, [100, 100, 10], method=initialization)
    for activation_name, (activation, derivative) in activations.items():
        layers = [(W.copy(), b.copy()) for W, b in initial_layers]
        funcs = [activation, activation, nnf.softmax]
        ders = [derivative, derivative, output_derivative]
        np.random.seed(123)
        start = time.perf_counter()
        layers = nnf.train_sgd_adam(
            Xtrain=X_train, Ytrain=Y_train, layers=layers,
            activation_funcs=funcs, activation_ders=ders,
            epochs=EPOCHS, batch_size=BATCH_SIZE,
            gamma=LEARNING_RATE, beta1=0.9, beta2=0.999,
            use_decaying_step_length=False,
            cost_der=nnf.cross_entropy_der,
            method="Ridge", lbd=RIDGE_LAMBDA,
        )
        runtime = time.perf_counter() - start
        accuracy = nnf.classification_accuracy(layers, X_test, y_test, funcs)
        activation_results.append({"Initialization": initialization,
            "Activation": activation_name, "Test accuracy": accuracy,
            "Training time": runtime})
        print(f"{initialization:7} + {activation_name:10}: {accuracy:.4f}")

names = list(activations)
M = np.zeros((len(names), len(initializations)))
for result in activation_results:
    M[names.index(result["Activation"]), initializations.index(result["Initialization"])] = result["Test accuracy"]

x = np.arange(len(names)); width = 0.24
fig, ax = plt.subplots(figsize=(9, 5.5))
for j, initialization in enumerate(initializations):
    bars = ax.bar(x + (j - 1) * width, M[:, j], width,
                  label=initialization, edgecolor="black")
    ax.bar_label(bars, fmt="%.3f", padding=3)
ax.axhline(0.10, color="gray", linestyle="--", label="Random guessing")
ax.set(title="Test accuracy by activation and initialization",
       xlabel="Activation", ylabel="Test accuracy")
ax.set_xticks(x, names); ax.set_ylim(0, 1.08)
ax.grid(axis="y", alpha=0.25); ax.legend()
fig.tight_layout(); plt.show()

## Experiment 2: Optimization method

**Fixed:** `784 → 200 → 200 → 10`, Leaky ReLU, He initialization, softmax, cross-entropy, and ridge training.

**Varied:** Plain SGD, RMSprop, and Adam. Every optimizer receives identical starting parameters and shuffle seeds.

In [ ]:
optimizer_names = ["Plain SGD", "RMSprop", "Adam"]
optimizer_results = []
funcs = [nnf.leaky_ReLU, nnf.leaky_ReLU, nnf.softmax]
ders = [nnf.leaky_ReLU_der, nnf.leaky_ReLU_der, output_derivative]
np.random.seed(42)
initial_layers = nnf.create_layers(784, [200, 200, 10], method="He")

for optimizer in optimizer_names:
    layers = [(W.copy(), b.copy()) for W, b in initial_layers]
    np.random.seed(123)
    start = time.perf_counter()
    common = dict(
        Xtrain=X_train, Ytrain=Y_train, layers=layers,
        activation_funcs=funcs, activation_ders=ders, epochs=EPOCHS,
        batch_size=BATCH_SIZE, gamma=LEARNING_RATE,
        use_decaying_step_length=False, cost_der=nnf.cross_entropy_der,
        method="Ridge", lbd=RIDGE_LAMBDA,
    )
    if optimizer == "Plain SGD":
        layers = nnf.train_sgd_plain(**common)
    elif optimizer == "RMSprop":
        layers = nnf.train_sgd_RMSprop(beta2=0.999, **common)
    else:
        layers = nnf.train_sgd_adam(beta1=0.9, beta2=0.999, **common)
    runtime = time.perf_counter() - start
    accuracy = nnf.classification_accuracy(layers, X_test, y_test, funcs)
    optimizer_results.append({"Optimizer": optimizer, "Test accuracy": accuracy,
                              "Training time": runtime})
    print(f"{optimizer:10}: accuracy={accuracy:.4f}, time={runtime:.2f}s")

labels = [r["Optimizer"] for r in optimizer_results]
accuracy = np.array([r["Test accuracy"] for r in optimizer_results])
runtime = np.array([r["Training time"] for r in optimizer_results])
x = np.arange(len(labels))
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
bars = axes[0].bar(x, accuracy, color="tab:orange", edgecolor="black")
axes[0].set(title="Test accuracy by optimizer", ylabel="Test accuracy")
axes[0].set_xticks(x, labels); axes[0].set_ylim(0, 1.08)
axes[0].bar_label(bars, fmt="%.3f"); axes[0].grid(axis="y", alpha=.25)
bars = axes[1].bar(x, runtime, color="tab:green", edgecolor="black")
axes[1].set(title="Training time by optimizer", ylabel="Seconds")
axes[1].set_xticks(x, labels)
axes[1].bar_label(bars, labels=[f"{v:.2f} s" for v in runtime])
axes[1].grid(axis="y", alpha=.25)
fig.tight_layout(); plt.show()

## Experiment 3: Model architecture

**Fixed:** Leaky ReLU, He initialization, RMSprop, softmax, cross-entropy, learning rate, batch size, epochs, and ridge training.

**Varied:** network depth, width, and layer ordering. The logistic-regression entry is a linear softmax baseline.

In [ ]:
architectures = {
    "Logistic regression": [10],
    "Single wide layer": [400, 10],
    "Increasing": [50, 125, 225, 10],
    "Decreasing": [225, 125, 50, 10],
    "Small-big-small": [100, 200, 100, 10],
    "Big-small-big": [195, 10, 195, 10],
    "Constant deep": [50, 50, 50, 50, 50, 50, 50, 50, 10],
}
architecture_results = []
trained_architecture_models = {}

for name, sizes in architectures.items():
    hidden_layers = len(sizes) - 1
    funcs = [nnf.leaky_ReLU] * hidden_layers + [nnf.softmax]
    ders = [nnf.leaky_ReLU_der] * hidden_layers + [output_derivative]
    np.random.seed(42)
    layers = nnf.create_layers(784, sizes, method="He")
    np.random.seed(123)
    start = time.perf_counter()
    layers = nnf.train_sgd_RMSprop(
        Xtrain=X_train, Ytrain=Y_train, layers=layers,
        activation_funcs=funcs, activation_ders=ders,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        gamma=LEARNING_RATE, beta2=0.999,
        use_decaying_step_length=False,
        cost_der=nnf.cross_entropy_der,
        method="Ridge", lbd=RIDGE_LAMBDA,
    )
    runtime = time.perf_counter() - start
    accuracy = nnf.classification_accuracy(layers, X_test, y_test, funcs)
    parameters = sum(W.size + b.size for W, b in layers)
    architecture_results.append({"Architecture": name, "Test accuracy": accuracy,
        "Training time": runtime, "Parameters": parameters})
    trained_architecture_models[name] = {"layers": layers, "activation_funcs": funcs}
    print(f"{name:22}: accuracy={accuracy:.4f}, time={runtime:.2f}s, parameters={parameters:,}")

labels = [r["Architecture"] for r in architecture_results]
accuracy = np.array([r["Test accuracy"] for r in architecture_results])
runtime = np.array([r["Training time"] for r in architecture_results])
x = np.arange(len(labels))
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
bars = axes[0].bar(x, accuracy, color="tab:orange", edgecolor="black")
axes[0].set(title="Test accuracy by architecture", ylabel="Test accuracy")
axes[0].set_xticks(x, labels, rotation=35, ha="right"); axes[0].set_ylim(0, 1.08)
axes[0].bar_label(bars, fmt="%.3f"); axes[0].grid(axis="y", alpha=.25)
bars = axes[1].bar(x, runtime, color="tab:green", edgecolor="black")
axes[1].set(title="Training runtime by architecture", ylabel="Seconds")
axes[1].set_xticks(x, labels, rotation=35, ha="right")
axes[1].bar_label(bars, labels=[f"{v:.2f} s" for v in runtime], fontsize=9)
axes[1].grid(axis="y", alpha=.25); axes[1].set_ylim(0, runtime.max() * 1.15)
fig.tight_layout(); plt.show()

## Experiment 4: ROC/AUC

Each trained architecture is compared with a macro-averaged one-vs-rest ROC curve. Confusion matrices are intentionally postponed until after cross-validation, so the tuned single-wide-layer model is used in that comparison.

In [ ]:
def macro_roc(y_true, scores):
    grid = np.linspace(0, 1, 1_000)
    mean_tpr = np.zeros_like(grid)
    for class_index in range(10):
        fpr, tpr, _ = roc_curve(y_true[:, class_index], scores[:, class_index])
        mean_tpr += np.interp(grid, fpr, tpr)
    mean_tpr /= 10
    mean_tpr[0], mean_tpr[-1] = 0.0, 1.0
    return grid, mean_tpr, auc(grid, mean_tpr)

roc_results = {}
for name, model in trained_architecture_models.items():
    scores = np.asarray(nnf.feed_forward(X_test, model["layers"], model["activation_funcs"]))
    fpr, tpr, area = macro_roc(Y_test, scores)
    roc_results[name] = {"FPR": fpr, "TPR": tpr, "AUC": area, "Scores": scores}

ranked = sorted(roc_results.items(), key=lambda item: item[1]["AUC"], reverse=True)
fig, ax = plt.subplots(figsize=(8, 5.5))
for rank, (name, result) in enumerate(ranked, start=1):
    ax.plot(result["FPR"], result["TPR"], linewidth=2,
            label=f"{rank}. {name} (AUC={result['AUC']:.6f})")
    print(f"{rank}. {name:<25} AUC={result['AUC']:.7f}")
ax.set(title="Macro-averaged ROC curves by architecture\nMNIST test set",
       xlabel="False positive rate", ylabel="True positive rate")
ax.set_xlim(0, .20); ax.set_ylim(.80, 1.005)
ax.grid(alpha=.25); ax.legend(loc="lower right", fontsize=10)
fig.tight_layout(); plt.show()

## Experiment 5: Cross-validation of learning rate and ridge penalty

A scikit-learn-compatible classifier wraps the custom single-wide-layer network. `StratifiedKFold` preserves class proportions, and `GridSearchCV` selects `gamma` and `lbd` using validation accuracy. The held-out MNIST test set is not used during model selection. After selection, `GridSearchCV(refit=True)` fits the best configuration on the full training set.

The standard error shown for each grid point is `std_test_score / sqrt(k)`, where `k` is the number of folds.

In [ ]:
X_train = np.ascontiguousarray(X_train, dtype=np.float32)
X_test = np.ascontiguousarray(X_test, dtype=np.float32)

print("X_train dtype:", X_train.dtype)
print("X_test dtype:", X_test.dtype)
print("Training data size:", X_train.nbytes / 1024**2, "MiB")

cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=2026,
)

# 3 learning-rate candidates
learning_rates = [
    0.0001,
    0.001,
    0.01,
]

# 3 ridge-penalty candidates
ridge_lambdas = [
    0.0,
    1e-5,
    1e-2,
]

net = nnf.NeuralNetClassifier(
    hidden=(400,),
    activation="leaky_relu",
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    seed=2026,
)

# 4 × 4 = 16 candidate combinations
param_grid = {
    "gamma": learning_rates,
    "lbd": ridge_lambdas,
}

search = GridSearchCV(
    estimator=net,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    return_train_score=True,
    n_jobs=1,
    refit=True,
    verbose=2,
)

search.fit(X_train, y_train)


# Extract the cross-validation results.
results = search.cv_results_

mean_accuracy = results["mean_test_score"].reshape(
    len(learning_rates),
    len(ridge_lambdas),
)

standard_error = (
    results["std_test_score"]
    / np.sqrt(cv.get_n_splits())
).reshape(
    len(learning_rates),
    len(ridge_lambdas),
)


print("Best parameters:", search.best_params_)
print(f"Best mean CV accuracy: {search.best_score_:.4f}")
print(
    "Tuned single-wide test accuracy: "
    f"{search.score(X_test, y_test):.4f}"
)


# GridSearchCV has refitted the best candidate on all training data.
best_single_wide = search.best_estimator_

# Replace the earlier untuned single-wide model.
trained_architecture_models["Single wide layer"] = {
    "layers": best_single_wide.layers_,
    "activation_funcs": best_single_wide.activation_funcs_,
}

# Calculate probabilities and ROC information for the tuned model.
tuned_single_wide_scores = best_single_wide.predict_proba(
    X_test
)

fpr, tpr, area = macro_roc(
    Y_test,
    tuned_single_wide_scores,
)

roc_results["Single wide layer"] = {
    "FPR": fpr,
    "TPR": tpr,
    "AUC": area,
    "Scores": tuned_single_wide_scores,
}

X_train dtype: float32
X_test dtype: float32
Training data size: 179.443359375 MiB


NameError: name 'NeuralNetClassifier' is not defined

In [ ]:

# Plot the 4 × 4 cross-validation results.
fig, ax = plt.subplots(figsize=(8, 6))

image = ax.imshow(
    mean_accuracy,
    aspect="auto",
    cmap="viridis",
)

ax.set_xticks(
    np.arange(len(ridge_lambdas)),
    [f"{value:g}" for value in ridge_lambdas],
)

ax.set_yticks(
    np.arange(len(learning_rates)),
    [f"{value:g}" for value in learning_rates],
)

ax.set(
    xlabel="Ridge penalty λ",
    ylabel="Learning rate γ",
    title="4-fold CV accuracy: single-wide-layer network",
)

for i in range(len(learning_rates)):
    for j in range(len(ridge_lambdas)):
        text_color = (
            "white"
            if mean_accuracy[i, j] < mean_accuracy.mean()
            else "black"
        )

        ax.text(
            j,
            i,
            (
                f"{mean_accuracy[i, j]:.3f}\n"
                f"±{standard_error[i, j]:.3f}"
            ),
            ha="center",
            va="center",
            color=text_color,
            fontsize=9,
        )

fig.colorbar(
    image,
    ax=ax,
    label="Mean validation accuracy",
)

fig.tight_layout()
plt.show()


# GridSearchCV has refitted the best candidate on all training data.
best_single_wide = search.best_estimator_

# Replace the earlier untuned single-wide model.
trained_architecture_models["Single wide layer"] = {
    "layers": best_single_wide.layers_,
    "activation_funcs": best_single_wide.activation_funcs_,
}

# Calculate probabilities and ROC information for the tuned model.
tuned_single_wide_scores = best_single_wide.predict_proba(
    X_test
)

fpr, tpr, area = macro_roc(
    Y_test,
    tuned_single_wide_scores,
)

roc_results["Single wide layer"] = {
    "FPR": fpr,
    "TPR": tpr,
    "AUC": area,
    "Scores": tuned_single_wide_scores,
}

## Confusion matrices after cross-validation

The tuned single-wide-layer network is compared with the decreasing architecture and logistic-regression baseline. The single-wide matrix therefore uses the best learning rate and ridge penalty selected without looking at the test labels.

In [ ]:
selected = ["Single wide layer", "Decreasing", "Logistic regression"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, name in zip(axes, selected):
    predicted = np.argmax(roc_results[name]["Scores"], axis=1)
    ConfusionMatrixDisplay.from_predictions(
        y_test, predicted, labels=np.arange(10), display_labels=np.arange(10),
        cmap="Blues", colorbar=False, values_format="d", ax=ax,
    )
    ax.set_title(f"{name}\nTest accuracy={np.mean(predicted == y_test):.3f}")
    ax.set_xlabel("Predicted digit")
    ax.set_ylabel("True digit")
fig.suptitle("Confusion matrices by architecture on MNIST")
fig.tight_layout(); plt.show()

## Predicted probability for digit 8

The same three architectures are compared using their predicted probabilities for digit 8.

In [ ]:
selected_architectures = ["Single wide layer", "Decreasing", "Logistic regression"]
positive_class = 8
threshold = 0.5
y_true_binary = (y_test == positive_class).astype(int)
bins = np.linspace(0.0, 1.0, 31)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
for ax, architecture_name in zip(axes, selected_architectures):
    scores_for_eight = roc_results[architecture_name]["Scores"][:, positive_class]
    ax.hist(scores_for_eight[y_true_binary == 0], bins=bins, alpha=0.65, label="Not 8")
    ax.hist(scores_for_eight[y_true_binary == 1], bins=bins, alpha=0.65, label="8")
    ax.axvline(threshold, color="black", linestyle="--", linewidth=1.3, label="t = 0.5")
    ax.set_yscale("log")
    ax.set_xlabel("Predicted probability of digit 8")
    ax.set_title(architecture_name)
    ax.grid(axis="y", alpha=0.20)
    ax.legend()
axes[0].set_ylabel("Number of images, log scale")
fig.suptitle("Distribution of predicted probabilities for digit 8", fontsize=14)
fig.tight_layout(); plt.show()

## Notes

- Full mode is computationally expensive because the grid alone performs 25 parameter combinations × 5 folds, followed by one refit.
- Use `USE_SMALL_DATASET = True` to verify the workflow before a full run.
- Cross-validation uses only the training set. The test set is evaluated only after the best single-wide configuration has been selected.
- If your local `neural_network_functions.py` uses a different ridge keyword than `lbd`, update that keyword consistently; this notebook assumes the provided training API accepts `method="Ridge"` and `lbd=...`.